# Asymmetric Loss for conservative prediction

** Purpuse: minimize the underestimation risk - grid operators prefer over-provisioning**

* Option 1 — Post-processing (no retraining) - Pros: instant, no retraining. Cons: static offset, doesn't adapt to context.
* Option 2 — Quantile regression (retrain)

In [ ]:
# Option 1 — Post-processing (no retraining)

# On a validation set, collect residuals: residual = actual - predicted
# If residuals skew positive (you under-predict), add a percentile offset
residuals = y_val - model.predict(X_val)
offset = np.percentile(residuals, 75)   # moves ~75% of predictions above actual
preds_conservative = model.predict(X) + offset

In [ ]:
# Option 2 — Quantile regression (retrain)

# LightGBM — predict 70th percentile instead of mean
import lightgbm as lgb
model_conservative = lgb.LGBMRegressor(
    objective='quantile',
    alpha=0.70,        # tune: 0.6–0.8
    **other_params
)

# XGBoost
import xgboost as xgb
model_conservative = xgb.XGBRegressor(
    objective='reg:quantileerror',
    quantile_alpha=0.70,
    **other_params
)

# Random Forest — predict upper bound of confidence interval
from sklearn.quantile-forest import  RandomForestQuantileRegressor
model_conservative = RandomForestQuantileRegressor(
    quantile=0.70,
    **other_params
)



In [2]:
import sys
import os

# Add the src directory to the system path to allow importing custom modules
sys.path.insert(0, os.path.abspath('../src'))

import warnings
warnings.filterwarnings('ignore')

# Enable autoreload to automatically reload modules when they are edited
%load_ext autoreload
%autoreload 2

import numpy as np
import pickle
import pandas as pd

from fetch_prepare_data import *
from train_model_predict import *

# Set a consistent style for all plots
import matplotlib.pyplot as plt
plt.rcParams.update({
    'axes.grid':      True,
    'grid.color':     '#DCDCDC',
    'grid.linewidth': 0.5,
    'grid.linestyle': '-',
    'axes.axisbelow': True,
    'axes.facecolor': 'white',
    'font.family':    'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titlepad':  13,
    'axes.labelsize': 10,
    'axes.labelpad':  8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'legend.frameon':    True,
    'legend.facecolor':  'white',
    'legend.edgecolor':  '#DCDCDC',
    'legend.framealpha': 1.0,
    'legend.fontsize':   9,
})


In [ ]:
# tune LightGBM conservatively with Bayesian optimization using Optuna 
from lightgbm import LGBMRegressor
import pandas as pd
from fetch_prepare_data import *
from train_model_predict import *

with open('../data/processed/energy_weather_data_for_modeling.pkl', 'rb') as f:
    df_for_modeling = pd.read_pickle(f)
#df_for_modeling = prepare_data_for_modeling()

features_train, target_train, features_test, target_test = train_test_split_by_date(df_for_modeling, 
                                                                                    date_column='time',
                                                                                    target_column='EnergyDemand', 
                                                                                    split_date='2025-01-01')

# define continuous parameter grid for LightGBM
param_lgbm_continuous = { 
    'n_estimators': (50, 500),  # range for n_estimators
    'learning_rate': (0.01, 0.3),  # range for learning_rate
    'max_depth': (3, 15)  # range for max_depth
}

model_lgbm_conservative =LGBMRegressor(    
    objective='quantile',
    alpha=0.90, # the higher the alpha, the more conservative - less under-prediction, more over-prediction
    random_state=42, 
    force_col_wise=True)
best_model_lgbm, best_params_lgbm = tune_model_bayesian(
    model_pipeline=model_lgbm_conservative, 
    in_param_bayes=param_lgbm_continuous,
    in_features_train=features_train, 
    in_target_train=target_train)
print(f"Best hyperparameters: {best_params_lgbm}")

print()

prediction_lgbm = best_model_lgbm.predict(features_test)

print_scores('LightGBM', target_test, prediction_lgbm)  

save_model_to_pickle(best_model_lgbm, '../models/best_lgbm_model_bayesian_conservative.pkl')


[LightGBM] [Info] Total Bins 3900
[LightGBM] [Info] Number of data points in the train set: 52416, number of used features: 26
[LightGBM] [Info] Start training from score 64263.046875
Best hyperparameters: OrderedDict({'learning_rate': 0.19828981519978864, 'max_depth': 15, 'n_estimators': 418})

-------------------- scoring --------------------
model                  MAE       RMSE       R²
LightGBM            1283.00    1654.03     0.97


In [5]:
# What fraction of predictions are <= actual?
coverage = (prediction_lgbm <= target_test.values).mean()
print(f"Coverage (pred <= actual): {coverage:.1%}")   # should be ~70% for alpha=0.70

# Also check mean over-prediction (cost of conservatism)
overpred = (prediction_lgbm - target_test.values)
print(f"Mean bias:  {overpred.mean():+,.0f} MWh")
print(f"Under-preds: {(overpred < 0).sum()} / {len(overpred)} hours")

Coverage (pred <= actual): 42.7%
Mean bias:  +222 MWh
Under-preds: 2800 / 6553 hours
